# Colab T4 Segment Embedding Aspect Discovery

런타임 → 런타임 유형 변경 → 하드웨어 가속기 GPU 선택

권장 GPU: T4

이 노트북은 segment 임베딩, UMAP/HDBSCAN 또는 KMeans 클러스터링, c-TF-IDF 키워드 추출, 대표 segment 추출을 수행합니다.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!pip -q install -U sentence-transformers umap-learn hdbscan scikit-learn pandas numpy tqdm

왼쪽 파일 패널에 `segments_for_embedding_colab.csv`를 업로드하거나 Google Drive를 마운트한 뒤 `INPUT_PATH`를 수정하세요.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

INPUT_PATH = '/content/segments_for_embedding_colab.csv'
OUTPUT_DIR = Path('/content/colab_aspect_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(INPUT_PATH, encoding='utf-8-sig')
df['segment_text_for_model'] = df.get('segment_text_for_model', df['segment_text']).fillna('').astype(str)
print(df.shape)
df.head()

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_CANDIDATES = [
    'jhgan/ko-sroberta-multitask',
    'BM-K/KoSimCSE-roberta-multitask',
    'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    'intfloat/multilingual-e5-base',
]
BATCH_SIZE = 128
model = None
loaded_name = None
for name in MODEL_CANDIDATES:
    try:
        model = SentenceTransformer(name, device='cuda' if torch.cuda.is_available() else 'cpu')
        loaded_name = name
        break
    except Exception as exc:
        print('failed:', name, exc)
if model is None:
    raise RuntimeError('No embedding model could be loaded')
print('loaded:', loaded_name)

In [ ]:
texts = df['segment_text_for_model'].tolist()
batch_size = BATCH_SIZE
while True:
    try:
        embeddings = model.encode(texts, batch_size=batch_size, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
        break
    except RuntimeError as exc:
        if batch_size <= 32:
            raise
        batch_size //= 2
        torch.cuda.empty_cache()
        print('retry batch_size:', batch_size)
np.save(OUTPUT_DIR / 'segment_embeddings.npy', embeddings)
print(embeddings.shape)

In [ ]:
import umap
from sklearn.cluster import KMeans

reducer = umap.UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
reduced = reducer.fit_transform(embeddings)
try:
    import hdbscan
    clusterer = hdbscan.HDBSCAN(min_cluster_size=10, metric='euclidean', prediction_data=True)
    labels = clusterer.fit_predict(reduced)
    probs = getattr(clusterer, 'probabilities_', np.ones(len(labels)))
    method = 'hdbscan'
    if len(set(labels)) <= 1:
        raise ValueError('HDBSCAN made no useful clusters')
except Exception as exc:
    print('HDBSCAN fallback to KMeans:', exc)
    n_clusters = min(10, max(2, len(df) // 50))
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init='auto')
    labels = kmeans.fit_predict(reduced)
    probs = np.ones(len(labels))
    method = 'kmeans'

clusters = df.copy()
clusters['cluster_id'] = labels
clusters['cluster_probability'] = probs
clusters['cluster_method'] = method
clusters.to_csv(OUTPUT_DIR / 'segment_embedding_clusters.csv', index=False, encoding='utf-8-sig')
clusters[['segment_id','cluster_id','cluster_probability','segment_text']].head()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

keyword_rows = []
rep_rows = []
vectorizer = TfidfVectorizer(max_features=5000, token_pattern=r'(?u)\\b[0-9A-Za-z가-힣]{2,}\\b')
for cluster_id, group in clusters[clusters['cluster_id'] != -1].groupby('cluster_id'):
    texts_cluster = group['segment_text_for_model'].fillna('').astype(str).tolist()
    if len(texts_cluster) < 2:
        continue
    mat = vectorizer.fit_transform(texts_cluster)
    terms = vectorizer.get_feature_names_out()
    weights = np.asarray(mat.mean(axis=0)).ravel()
    for rank, idx in enumerate(weights.argsort()[::-1][:15], start=1):
        keyword_rows.append({'cluster_id': cluster_id, 'rank': rank, 'keyword': terms[idx], 'weight': float(weights[idx])})
    reps = group.sort_values('cluster_probability', ascending=False).head(10)
    for rank, (_, row) in enumerate(reps.iterrows(), start=1):
        rep_rows.append({'cluster_id': cluster_id, 'rank': rank, 'segment_id': row['segment_id'], 'segment_text': row['segment_text']})

keywords = pd.DataFrame(keyword_rows)
reps = pd.DataFrame(rep_rows)
mapping = keywords.groupby('cluster_id')['keyword'].apply(lambda s: ', '.join(s.head(10))).reset_index(name='top_keywords')
mapping['aspect_name_candidate'] = ''
mapping['human_review_memo'] = ''
keywords.to_csv(OUTPUT_DIR / 'aspect_cluster_keywords.csv', index=False, encoding='utf-8-sig')
reps.to_csv(OUTPUT_DIR / 'aspect_representative_segments.csv', index=False, encoding='utf-8-sig')
mapping.to_csv(OUTPUT_DIR / 'aspect_name_mapping_template.csv', index=False, encoding='utf-8-sig')
pd.DataFrame([{'embedding_model': loaded_name, 'batch_size': batch_size, 'cluster_method': method, 'segment_count': len(df)}]).to_csv(OUTPUT_DIR / 'embedding_run_log.csv', index=False, encoding='utf-8-sig')
list(OUTPUT_DIR.glob('*.csv'))

생성된 CSV를 다운로드해서 로컬 `C:\every_time\outputs\colab_results`에 넣으세요.